In [11]:
import time
import pandas as pd
import numpy as np

from pylsl import StreamInfo, StreamOutlet

In [12]:
LOG_FILE = "../data/04-06/s1/s2_log.csv"
EEG_FILE = "../data/04-06/s1/s1_eeg.txt"

SAMPLE_RATE = 125
N_CHANNELS = 16

In [13]:
# ==========================================
# FIND HEADER
# ==========================================

with open(EEG_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()

header_idx = None

for i, line in enumerate(lines):
    if "Sample Index" in line:
        header_idx = i
        break

if header_idx is not None:
    print("Header found at line:", header_idx)
else:
    print("Header not found")

eeg_df = pd.read_csv(
    EEG_FILE,
    skiprows=header_idx
)

# Clean column names
eeg_df.columns = [c.strip() for c in eeg_df.columns]

print("Shape:", eeg_df.shape)

eeg_channels = [f"EXG Channel {i}" for i in range(16)]

eeg_data = eeg_df[eeg_channels].values

print("EEG Shape:", eeg_data.shape)

eeg_time = pd.to_datetime(
    eeg_df["Timestamp (Formatted)"]
)

eeg_time.head()
# type(eeg_time.iloc[0])

Header found at line: 4
Shape: (57102, 33)
EEG Shape: (57102, 16)


0   2026-06-04 17:11:48.594
1   2026-06-04 17:11:48.594
2   2026-06-04 17:11:48.697
3   2026-06-04 17:11:48.704
4   2026-06-04 17:11:48.711
Name: Timestamp (Formatted), dtype: datetime64[us]

In [14]:
log_df = pd.read_csv(LOG_FILE)

record_date = eeg_time.iloc[0].date()

log_df["DateTime"] = pd.to_datetime(
    str(record_date) + " " + log_df["Time"]
)

log_df.head()

,Time,Experiment,Phase,Event,Detail,Action,DateTime
0,17:15:55.548,BCI,Test3D,LSL_Connected,BCIBackend,BCI_Stream_Open,2026-06-04 17:15:55.548
1,17:15:55.570,BCI,Test3D,Experiment_Start,BCI,NaN,2026-06-04 17:15:55.570
2,17:15:55.570,BCI,TrainBCI,Phase_Start,TrainBCI,NaN,2026-06-04 17:15:55.570
3,17:15:55.602,BCI,TrainBCI,Hover_Enter,Button: StartTrainBCI,Hovering,2026-06-04 17:15:55.602
4,17:15:55.604,BCI,TrainBCI,Dwell_Start,Button: StartTrainBCI,Dwell_Started,2026-06-04 17:15:55.604


In [15]:
common_start = log_df["DateTime"].iloc[0]

eeg_offsets = (eeg_time - common_start).dt.total_seconds()
marker_offsets = (log_df["DateTime"] - common_start).dt.total_seconds()

In [16]:
valid_eeg = eeg_offsets >= 0

eeg_data = eeg_data[valid_eeg]
eeg_offsets = eeg_offsets[valid_eeg]

In [17]:
# resolve_byprop("type", "EEG")
# resolve_byprop("type", "Markers")

In [18]:
eeg_info = StreamInfo(
    name="obci_eeg1",
    type="EEG",
    channel_count=16,
    nominal_srate=125,
    channel_format="float32",
    source_id="replay_eeg"
)

eeg_outlet = StreamOutlet(eeg_info)

In [19]:
marker_info = StreamInfo(
    name="Unity_Markers",
    type="Markers",
    channel_count=1,
    nominal_srate=0,
    channel_format="string",
    source_id="replay_markers"
)

marker_outlet = StreamOutlet(marker_info)

In [20]:
from PIL import TiffImagePlugin
print("Replay starts in 5 sec...")

time.sleep(5)

marker_idx = 0

t0 = time.perf_counter()

for i in range(len(eeg_data)):

    elapsed = time.perf_counter() - t0

    while (
        marker_idx < len(log_df)
        and marker_offsets.iloc[marker_idx] <= elapsed
    ):
        eve = str(
            log_df.iloc[marker_idx]["Event"]
        )
        det = str(
            log_df.iloc[marker_idx]["Detail"]
        )
        action = str(
            log_df.iloc[marker_idx]["Action"]
        )

        marker_outlet.push_sample([action])

        print(
            f"Marker {marker_idx}:",
            eve, det, action
        )

        marker_idx += 1

    eeg_outlet.push_sample(
        eeg_data[i].tolist()
    )

    target_time = eeg_offsets.iloc[i]

    while (
        time.perf_counter() - t0
        < target_time
    ):
        time.sleep(0.0005)

print("Replay complete")

Replay starts in 5 sec...
Marker 0: LSL_Connected BCIBackend BCI_Stream_Open
Marker 1: Experiment_Start BCI nan
Marker 2: Phase_Start TrainBCI nan
Marker 3: Hover_Enter Button: StartTrainBCI Hovering
Marker 4: Dwell_Start Button: StartTrainBCI Dwell_Started
Marker 5: Eye_Closed eye_tracking Eye_Closed
Marker 6: Eye_Opened eye_tracking Eye_Opened
Marker 7: Hover_Exit Button: StartTrainBCI Hover_Exit
Marker 8: Hover_Enter Button: StartTrainBCI Hovering
Marker 9: Hover_Exit Button: StartTrainBCI Hover_Exit
Marker 10: Hover_Enter Button: StartTrainBCI Hovering
Marker 11: Hover_Exit Button: StartTrainBCI Hover_Exit
Marker 12: Hover_Enter Button: StartTrainBCI Hovering
Marker 13: Hover_Exit Button: StartTrainBCI Hover_Exit
Marker 14: Hover_Enter Button: StartTrainBCI Hovering
Marker 15: Eye_Closed eye_tracking Eye_Closed
Marker 16: Eye_Opened eye_tracking Eye_Opened
Marker 17: Eye_Closed eye_tracking Eye_Closed
Marker 18: Eye_Opened eye_tracking Eye_Opened
Marker 19: Dwell Button: StartTrain

KeyboardInterrupt: 